# Week 6：LOF 建模（Gavin / Jiahao）

本 notebook 完成 Gavin 在 Week 6 的任务：基于 Lee 输出的共享 ML-ready dataset 建立 Local Outlier Factor（LOF）模型，测试多个 `n_neighbors` 参数，并导出 LOF anomaly score 和 anomaly label。

本 notebook 只使用 Lee 在 `feature_list.json` 中列出的 `model_feature_columns`。因此，`timestamp`、`zone` 等 identifier，以及 v3 statistical baseline label，都不会进入 LOF 模型训练。

## Cell 1：配置和导入库

这个 cell 设置文件路径、模型参数，并导入 notebook 会使用的 Python 库。`SAMPLE_SIZE = None` 表示 LOF 使用 Lee 的完整数据集，而不是抽样数据。本 notebook 测试一个 `3 x 3` 参数组合：3 个 `n_neighbors` 和 3 个 `contamination`。

In [1]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.neighbors import LocalOutlierFactor

os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count() or 1))

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "ml" else Path.cwd()
ML_READY_DIR = PROJECT_DIR / "outputs" / "floor6_zone1245_ml_ready"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "floor6_zone1245_lof_modelling_gavin"

ML_READY_PATH = ML_READY_DIR / "floor6_zone1245_ml_ready_2018_jul01_sep01.csv.gz"
FEATURE_LIST_PATH = ML_READY_DIR / "feature_list.json"

RANDOM_STATE = 42
SAMPLE_SIZE = None
NEIGHBOR_SETTINGS = [20, 50, 100]
CONTAMINATION_SETTINGS = [0.03, 0.05, 0.09]
SELECTED_N_NEIGHBORS = 100
SELECTED_CONTAMINATION = 0.05
N_JOBS = -1

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"项目目录: {PROJECT_DIR}")
print(f"ML-ready 数据集: {ML_READY_PATH}")
print(f"输出目录: {OUTPUT_DIR}")

项目目录: d:\IRP\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis
ML-ready 数据集: d:\IRP\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\floor6_zone1245_ml_ready\floor6_zone1245_ml_ready_2018_jul01_sep01.csv.gz
输出目录: d:\IRP\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\floor6_zone1245_lof_modelling_gavin


## Cell 2：读取 Lee 的 ML-ready 数据

这个 cell 读取 Lee 的共享数据集和 feature manifest。它会检查所有模型特征是否存在、是否都是数值型，并确认后续比较需要的参考字段是否保留在数据中。

In [2]:
if not ML_READY_PATH.exists():
    raise FileNotFoundError(
        f"找不到 {ML_READY_PATH}。请先运行 Lee 的 ML_Data_Preparation_Feature_Engineering.ipynb。"
    )

feature_manifest = json.loads(FEATURE_LIST_PATH.read_text(encoding="utf-8"))
model_features = feature_manifest["model_feature_columns"]

reference_columns = [
    "timestamp",
    "floor",
    "zone",
    "hour",
    "operating_period",
    "baseline_anomaly_label",
    "baseline_anomaly_count",
]

ml_ready = pd.read_csv(ML_READY_PATH, parse_dates=["timestamp"])

missing_features = sorted(set(model_features) - set(ml_ready.columns))
missing_reference = sorted(set(reference_columns) - set(ml_ready.columns))
if missing_features or missing_reference:
    raise ValueError(
        f"缺少模型特征: {missing_features}; 缺少参考字段: {missing_reference}"
    )

non_numeric_features = ml_ready[model_features].select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric_features:
    raise TypeError(f"LOF 需要数值型特征。非数值型字段: {non_numeric_features}")

if ml_ready[model_features].isna().any().any():
    raise ValueError("模型特征中存在 NaN。请重新检查 Lee 的 preprocessing 输出。")

if not np.isfinite(ml_ready[model_features].to_numpy(dtype=np.float32)).all():
    raise ValueError("模型特征中存在 infinite value。请重新检查 Lee 的 preprocessing 输出。")

baseline_rate = float(ml_ready["baseline_anomaly_label"].mean())

print(f"读取行数: {len(ml_ready):,}")
print(f"模型特征数量: {len(model_features)}")
print(f"用于比较的 v3 baseline 异常比例: {baseline_rate:.4%}")
print(f"LOF contamination 参数: {CONTAMINATION_SETTINGS}")
ml_ready[reference_columns].head()

读取行数: 362,864
模型特征数量: 42
用于比较的 v3 baseline 异常比例: 9.0329%
LOF contamination 参数: [0.03, 0.05, 0.09]


,timestamp,floor,zone,hour,operating_period,baseline_anomaly_label,baseline_anomaly_count
0,2018-07-01 00:04:00,6,1,0,weekend,0,0
1,2018-07-01 00:05:00,6,1,0,weekend,0,0
2,2018-07-01 00:06:00,6,1,0,weekend,0,0
3,2018-07-01 00:07:00,6,1,0,weekend,0,0
4,2018-07-01 00:08:00,6,1,0,weekend,0,0


## Cell 3：准备 LOF 建模数据

这个 cell 准备 LOF 使用的数据。因为 `SAMPLE_SIZE = None`，所以 notebook 会使用 Lee 的完整 ML-ready 数据。这里仍然保留分层抽样函数，方便之后如果运行时间太长，可以快速改成抽样实验。

In [3]:
def make_stratified_sample(frame: pd.DataFrame, sample_size: int | None) -> pd.DataFrame:
    if sample_size is None or sample_size >= len(frame):
        return frame.copy()

    strata = (
        frame["zone"].astype(str)
        + "_"
        + frame["operating_period"].astype(str)
        + "_"
        + frame["baseline_anomaly_label"].astype(str)
    )

    if strata.value_counts().min() < 2:
        return frame.sample(n=sample_size, random_state=RANDOM_STATE).copy()

    splitter = StratifiedShuffleSplit(
        n_splits=1,
        train_size=sample_size,
        random_state=RANDOM_STATE,
    )
    sample_index, _ = next(splitter.split(frame, strata))
    return frame.iloc[sample_index].sort_values("timestamp").copy()


lof_data = make_stratified_sample(ml_ready, SAMPLE_SIZE)
X = lof_data[model_features].to_numpy(dtype=np.float32)

sample_check = pd.DataFrame(
    {
        "dataset": ["Lee 完整数据集", "LOF 建模数据"],
        "rows": [len(ml_ready), len(lof_data)],
        "baseline_anomaly_rate": [ml_ready["baseline_anomaly_label"].mean(), lof_data["baseline_anomaly_label"].mean()],
        "zones": [ml_ready["zone"].nunique(), lof_data["zone"].nunique()],
        "operating_periods": [ml_ready["operating_period"].nunique(), lof_data["operating_period"].nunique()],
    }
)
sample_check

,dataset,rows,baseline_anomaly_rate,zones,operating_periods
0,Lee 完整数据集,362864,0.090329,4,6
1,LOF 建模数据,362864,0.090329,4,6


## Cell 4：训练 LOF 并测试 `n_neighbors` 与 `contamination`

这个 cell 会对每个 `n_neighbors` 拟合一次 LOF，然后在同一组 LOF score 排名上应用 3 个 `contamination` 阈值。这样既保留 `3 x 3` 参数比较，又避免对同一个 `n_neighbors` 重复计算邻域密度。这里把 LOF score 转换成“数值越高越异常”的方向。v3 baseline label 只用于结果比较，不作为模型特征。

In [4]:
def contamination_suffix(contamination: float) -> str:
    return f"c{int(round(contamination * 1000)):03d}"


def labels_from_score(score: np.ndarray, contamination: float) -> np.ndarray:
    anomaly_count = int(round(len(score) * contamination))
    if anomaly_count <= 0:
        return np.zeros(len(score), dtype=int)
    ranked_index = np.argsort(score)[::-1]
    labels = np.zeros(len(score), dtype=int)
    labels[ranked_index[:anomaly_count]] = 1
    return labels


results = lof_data[reference_columns].copy()
parameter_rows = []

for n_neighbors in NEIGHBOR_SETTINGS:
    if n_neighbors >= len(lof_data):
        raise ValueError(f"n_neighbors={n_neighbors} 必须小于建模数据行数。")

    start_time = time.time()
    model = LocalOutlierFactor(
        n_neighbors=n_neighbors,
        contamination="auto",
        n_jobs=N_JOBS,
    )
    model.fit_predict(X)
    fit_seconds = time.time() - start_time

    score_column = f"lof_score_k{n_neighbors}"
    results[score_column] = -model.negative_outlier_factor_

    for contamination in CONTAMINATION_SETTINGS:
        suffix = f"k{n_neighbors}_{contamination_suffix(contamination)}"
        label_column = f"lof_label_{suffix}"
        results[label_column] = labels_from_score(results[score_column].to_numpy(), contamination)

        lof_count = int(results[label_column].sum())
        baseline_count = int(results["baseline_anomaly_label"].sum())
        overlap_count = int(((results[label_column] == 1) & (results["baseline_anomaly_label"] == 1)).sum())
        union_count = int(((results[label_column] == 1) | (results["baseline_anomaly_label"] == 1)).sum())
        agreement_count = int((results[label_column] == results["baseline_anomaly_label"]).sum())

        parameter_rows.append(
            {
                "n_neighbors": n_neighbors,
                "contamination": contamination,
                "rows_modelled": len(results),
                "lof_anomaly_count": lof_count,
                "lof_anomaly_rate": lof_count / len(results),
                "baseline_anomaly_count": baseline_count,
                "baseline_anomaly_rate": baseline_rate,
                "baseline_overlap_count": overlap_count,
                "baseline_overlap_share_of_lof": overlap_count / lof_count if lof_count else np.nan,
                "jaccard_with_baseline": overlap_count / union_count if union_count else np.nan,
                "method_agreement_rate": agreement_count / len(results),
                "label_column": label_column,
                "score_column": score_column,
                "fit_seconds": fit_seconds,
            }
        )

parameter_summary = pd.DataFrame(parameter_rows)
parameter_summary

c:\Users\jh\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] 系统找不到指定的文件。
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\jh\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\jh\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\jh\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\jh\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                      

,n_neighbors,contamination,rows_modelled,lof_anomaly_count,lof_anomaly_rate,baseline_anomaly_count,baseline_anomaly_rate,baseline_overlap_count,baseline_overlap_share_of_lof,jaccard_with_baseline,method_agreement_rate,label_column,score_column,fit_seconds
0,20,0.03,362864,10886,0.030000,32777,0.090329,1216,0.111703,0.028647,0.886373,lof_label_k20_c030,lof_score_k20,91.706173
1,20,0.05,362864,18143,0.049999,32777,0.090329,1997,0.110070,0.040819,0.870679,lof_label_k20_c050,lof_score_k20,91.706173
2,20,0.09,362864,32658,0.090001,32777,0.090329,3513,0.107569,0.056733,0.839033,lof_label_k20_c090,lof_score_k20,91.706173
3,50,0.03,362864,10886,0.030000,32777,0.090329,1099,0.100955,0.025820,0.885729,lof_label_k50_c030,lof_score_k50,99.360661
4,50,0.05,362864,18143,0.049999,32777,0.090329,1632,0.089952,0.033112,0.868667,lof_label_k50_c050,lof_score_k50,99.360661
5,50,0.09,362864,32658,0.090001,32777,0.090329,2636,0.080715,0.041975,0.834200,lof_label_k50_c090,lof_score_k50,99.360661
6,100,0.03,362864,10886,0.030000,32777,0.090329,1788,0.164248,0.042699,0.889526,lof_label_k100_c030,lof_score_k100,104.640029
7,100,0.05,362864,18143,0.049999,32777,0.090329,2449,0.134983,0.050525,0.873170,lof_label_k100_c050,lof_score_k100,104.640029
8,100,0.09,362864,32658,0.090001,32777,0.090329,3805,0.116511,0.061739,0.840643,lof_label_k100_c090,lof_score_k100,104.640029


## Cell 5：选择主 LOF 结果并保存输出

这个 cell 选择 `n_neighbors = 100` 和 `contamination = 0.05` 作为主 LOF 设置。`k=100` 使用更平滑的邻域定义；`0.05` 比直接使用 v3 baseline 的异常比例更独立。这个 cell 会保存参数比较表、LOF anomaly 结果、top anomaly 预览，以及 JSON 摘要。

In [5]:
selected_suffix = f"k{SELECTED_N_NEIGHBORS}_{contamination_suffix(SELECTED_CONTAMINATION)}"
selected_label = f"lof_label_{selected_suffix}"
selected_score = f"lof_score_k{SELECTED_N_NEIGHBORS}"

results["lof_anomaly_label"] = results[selected_label]
results["lof_anomaly_score"] = results[selected_score]

parameter_summary_path = OUTPUT_DIR / "lof_parameter_comparison.csv"
results_filename = "lof_anomaly_results_full.csv.gz" if SAMPLE_SIZE is None else "lof_anomaly_results_sample.csv.gz"
results_path = OUTPUT_DIR / results_filename
top_anomalies_path = OUTPUT_DIR / "lof_top_anomalies.csv"
summary_path = OUTPUT_DIR / "lof_modelling_summary.json"

parameter_summary.to_csv(parameter_summary_path, index=False)
results.to_csv(results_path, index=False)
results.sort_values("lof_anomaly_score", ascending=False).head(100).to_csv(top_anomalies_path, index=False)

summary_payload = {
    "input_dataset": str(ML_READY_PATH.relative_to(PROJECT_DIR)),
    "feature_list": str(FEATURE_LIST_PATH.relative_to(PROJECT_DIR)),
    "output_results": str(results_path.relative_to(PROJECT_DIR)),
    "output_parameter_comparison": str(parameter_summary_path.relative_to(PROJECT_DIR)),
    "sample_size": None if SAMPLE_SIZE is None else int(SAMPLE_SIZE),
    "rows_available": int(len(ml_ready)),
    "rows_modelled": int(len(results)),
    "model_feature_count": int(len(model_features)),
    "n_neighbors_tested": NEIGHBOR_SETTINGS,
    "contamination_tested": CONTAMINATION_SETTINGS,
    "selected_n_neighbors": SELECTED_N_NEIGHBORS,
    "selected_contamination": SELECTED_CONTAMINATION,
    "baseline_anomaly_rate": baseline_rate,
    "baseline_label_used_as_feature": False,
    "identifier_columns_used_as_features": False,
}
summary_path.write_text(json.dumps(summary_payload, indent=2), encoding="utf-8")

print(f"已保存参数比较表: {parameter_summary_path}")
print(f"已保存 LOF anomaly 结果: {results_path}")
print(f"已保存 top anomaly 预览: {top_anomalies_path}")
print(f"已保存建模摘要: {summary_path}")

parameter_summary

已保存参数比较表: d:\IRP\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\floor6_zone1245_lof_modelling_gavin\lof_parameter_comparison.csv
已保存 LOF anomaly 结果: d:\IRP\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\floor6_zone1245_lof_modelling_gavin\lof_anomaly_results_full.csv.gz
已保存 top anomaly 预览: d:\IRP\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\floor6_zone1245_lof_modelling_gavin\lof_top_anomalies.csv
已保存建模摘要: d:\IRP\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\floor6_zone1245_lof_modelling_gavin\lof_modelling_summary.json


,n_neighbors,contamination,rows_modelled,lof_anomaly_count,lof_anomaly_rate,baseline_anomaly_count,baseline_anomaly_rate,baseline_overlap_count,baseline_overlap_share_of_lof,jaccard_with_baseline,method_agreement_rate,label_column,score_column,fit_seconds
0,20,0.03,362864,10886,0.030000,32777,0.090329,1216,0.111703,0.028647,0.886373,lof_label_k20_c030,lof_score_k20,91.706173
1,20,0.05,362864,18143,0.049999,32777,0.090329,1997,0.110070,0.040819,0.870679,lof_label_k20_c050,lof_score_k20,91.706173
2,20,0.09,362864,32658,0.090001,32777,0.090329,3513,0.107569,0.056733,0.839033,lof_label_k20_c090,lof_score_k20,91.706173
3,50,0.03,362864,10886,0.030000,32777,0.090329,1099,0.100955,0.025820,0.885729,lof_label_k50_c030,lof_score_k50,99.360661
4,50,0.05,362864,18143,0.049999,32777,0.090329,1632,0.089952,0.033112,0.868667,lof_label_k50_c050,lof_score_k50,99.360661
5,50,0.09,362864,32658,0.090001,32777,0.090329,2636,0.080715,0.041975,0.834200,lof_label_k50_c090,lof_score_k50,99.360661
6,100,0.03,362864,10886,0.030000,32777,0.090329,1788,0.164248,0.042699,0.889526,lof_label_k100_c030,lof_score_k100,104.640029
7,100,0.05,362864,18143,0.049999,32777,0.090329,2449,0.134983,0.050525,0.873170,lof_label_k100_c050,lof_score_k100,104.640029
8,100,0.09,362864,32658,0.090001,32777,0.090329,3805,0.116511,0.061739,0.840643,lof_label_k100_c090,lof_score_k100,104.640029


## Cell 6：按 zone、hour 和 operating period 汇总 LOF 异常

这个 cell 生成 Week 6 后续比较需要的基础分布表。这些表展示主 LOF 模型在不同 zone、hour 和 operating period 中检测到的异常分布。

In [6]:
def summarise_distribution(frame: pd.DataFrame, group_column: str) -> pd.DataFrame:
    summary = (
        frame.groupby(group_column, dropna=False)
        .agg(
            rows=("lof_anomaly_label", "size"),
            lof_anomalies=("lof_anomaly_label", "sum"),
            baseline_anomalies=("baseline_anomaly_label", "sum"),
            mean_lof_score=("lof_anomaly_score", "mean"),
        )
        .reset_index()
    )
    summary["lof_anomaly_rate"] = summary["lof_anomalies"] / summary["rows"]
    return summary.sort_values("lof_anomalies", ascending=False)


zone_summary = summarise_distribution(results, "zone")
hour_summary = summarise_distribution(results, "hour")
period_summary = summarise_distribution(results, "operating_period")

zone_summary.to_csv(OUTPUT_DIR / "lof_summary_by_zone.csv", index=False)
hour_summary.to_csv(OUTPUT_DIR / "lof_summary_by_hour.csv", index=False)
period_summary.to_csv(OUTPUT_DIR / "lof_summary_by_operating_period.csv", index=False)

display(zone_summary)
display(period_summary)
display(hour_summary.head(10))

,zone,rows,lof_anomalies,baseline_anomalies,mean_lof_score,lof_anomaly_rate
3,5,90716,5309,8514,1.094614,0.058523
0,1,90716,4796,12927,1.085591,0.052868
1,2,90716,4762,5475,1.080259,0.052493
2,4,90716,3276,5861,1.069497,0.036113


,operating_period,rows,lof_anomalies,baseline_anomalies,mean_lof_score,lof_anomaly_rate
5,weekend,103664,7192,3269,1.097143,0.069378
0,core_work,97200,4452,16245,1.080635,0.045802
1,evening_off,54000,2988,2767,1.085388,0.055333
2,night_off,75600,1907,5875,1.061612,0.025225
3,ramp_down,21600,897,3556,1.078706,0.041528
4,ramp_up,10800,707,1065,1.097773,0.065463


,hour,rows,lof_anomalies,baseline_anomalies,mean_lof_score,lof_anomaly_rate
19,19,15120,1113,704,1.097333,0.073611
9,9,15120,1036,2117,1.096830,0.068519
20,20,15120,1030,789,1.099670,0.068122
10,10,15120,984,2001,1.088337,0.065079
7,7,15120,957,1065,1.099466,0.063294
15,15,15120,932,2414,1.085331,0.061640
6,6,15120,921,690,1.091279,0.060913
22,22,15120,861,720,1.086427,0.056944
11,11,15120,849,1815,1.096967,0.056151
21,21,15120,844,719,1.083022,0.055820


## Cell 7：Gavin 交接说明

最后这个 cell 会打印给团队的简短交接说明。关键结果文件保存在 `outputs/floor6_zone1245_lof_modelling_gavin/`。对于 Jiahao/Gavin 的 Week 6 交付，最重要的文件是 `lof_parameter_comparison.csv` 和 `lof_anomaly_results_full.csv.gz`。

In [7]:
selected_row = parameter_summary.loc[
    (parameter_summary["n_neighbors"] == SELECTED_N_NEIGHBORS)
    & (parameter_summary["contamination"] == SELECTED_CONTAMINATION)
].iloc[0]

print("Gavin LOF 建模交接说明")
print(f"- Lee ML-ready 可用行数: {len(ml_ready):,}")
print(f"- 本 notebook 实际建模行数: {len(results):,}")
print(f"- 使用特征数量: {len(model_features)} 个 model_feature_columns")
print(f"- 测试的 n_neighbors: {NEIGHBOR_SETTINGS}")
print(f"- 测试的 contamination: {CONTAMINATION_SETTINGS}")
print(f"- 选定的 n_neighbors: {SELECTED_N_NEIGHBORS}")
print(f"- 选定的 contamination: {SELECTED_CONTAMINATION:.0%}")
print(f"- 选定参数下的 LOF anomalies: {int(selected_row['lof_anomaly_count']):,}")
print(f"- 与 v3 baseline 的 overlap: {int(selected_row['baseline_overlap_count']):,}")
print(f"- 结果文件夹: {OUTPUT_DIR}")

Gavin LOF 建模交接说明
- Lee ML-ready 可用行数: 362,864
- 本 notebook 实际建模行数: 362,864
- 使用特征数量: 42 个 model_feature_columns
- 测试的 n_neighbors: [20, 50, 100]
- 测试的 contamination: [0.03, 0.05, 0.09]
- 选定的 n_neighbors: 100
- 选定的 contamination: 5%
- 选定参数下的 LOF anomalies: 18,143
- 与 v3 baseline 的 overlap: 2,449
- 结果文件夹: d:\IRP\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\floor6_zone1245_lof_modelling_gavin
